# J9a : XGBoost + Optuna (Anti-Overfitting Pipeline)

**Features:** Mean-pooled MoCo (2048d) — le plus robust.

**Anti-Overfitting:**
- Optuna avec search space conservateur
- Patient-level StratifiedKFold CV
- Early stopping agressif
- Sauvegarde OOF + Test preds pour stacking


In [ ]:
# ============================================
# 1. MOUNT DRIVE & CONFIG
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/OWKIN_ML2/Data'
DRIVE_OUTPUT = '/content/drive/MyDrive/OWKIN_ML2/J9_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

TRAIN_MOCO = os.path.join(DRIVE_DATA, 'train_input', 'moco_features')
TEST_MOCO  = os.path.join(DRIVE_DATA, 'test_input', 'moco_features')
TRAIN_LABELS = os.path.join(DRIVE_DATA, 'train_output.csv')
META_TRAIN = os.path.join(DRIVE_DATA, 'supplementary_data', 'train_metadata.csv')
META_TEST  = os.path.join(DRIVE_DATA, 'supplementary_data', 'test_metadata.csv')

for p in [TRAIN_MOCO, TEST_MOCO, TRAIN_LABELS, META_TRAIN]:
    assert os.path.exists(p), f'NOT FOUND: {p}'
print('All paths OK ✅')


In [ ]:
import numpy as np
import pandas as pd
import json, shutil
from tqdm.notebook import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

!pip install -q xgboost
import xgboost as xgb
print('Imports OK ✅')


In [ ]:
# ============================================
# 2. DATA LOADING : Mean-Pool MoCo Features
# ============================================
df_meta_train = pd.read_csv(META_TRAIN)
df_meta_test  = pd.read_csv(META_TEST)
df_labels = pd.read_csv(TRAIN_LABELS)
df_train = df_meta_train.merge(df_labels, on='Sample ID')

def load_mean_features(sample_ids, moco_dir):
    features = []
    for sid in tqdm(sample_ids, desc='Loading'):
        data = np.load(os.path.join(moco_dir, sid))
        feats = data[:, 3:]  # drop (x, y, zoom)
        features.append(np.mean(feats, axis=0))  # Mean pool → 2048d
    return np.array(features)

X_train = load_mean_features(df_train['Sample ID'].values, TRAIN_MOCO)
X_test  = load_mean_features(df_meta_test['Sample ID'].values, TEST_MOCO)
y_train = df_train['Target'].values
patients_train = df_train['Patient ID'].values

# Patient-level CV setup
patients_unique = np.unique(patients_train)
y_patient = np.array([int(np.mean(y_train[patients_train == p])) for p in patients_unique])

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Patients uniques: {len(patients_unique)} | Positive rate: {y_train.mean():.2%}')


In [ ]:
# ============================================
# 3. OPTUNA : Conservative XGBoost Tuning
# ============================================
N_FOLDS = 5
N_OPTUNA_TRIALS = 40

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'gpu_hist',
        'verbosity': 0,
        'seed': 42,
        # ====== CONSERVATIVE SEARCH SPACE ======
        'max_depth':        trial.suggest_int('max_depth', 2, 5),          # SHALLOW
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.05, log=True),  # LOW LR
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 10, 50),  # HIGH → anti-overfit
        'subsample':        trial.suggest_float('subsample', 0.5, 0.8),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 0.4),  # LOW → diversity
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),   # L1
        'reg_lambda':       trial.suggest_float('reg_lambda', 1.0, 50.0, log=True),  # L2 STRONG
        'gamma':            trial.suggest_float('gamma', 0.1, 5.0, log=True),  # min split loss
    }
    
    aucs = []
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=42)
    
    for train_idx, val_idx in skf.split(patients_unique, y_patient):
        train_mask = np.isin(patients_train, patients_unique[train_idx])
        val_mask   = np.isin(patients_train, patients_unique[val_idx])
        
        X_tr, y_tr = X_train[train_mask], y_train[train_mask]
        X_val, y_val = X_train[val_mask], y_train[val_mask]
        
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        preds = model.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val, preds))
    
    return np.mean(aucs)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\n✅ Best CV AUC: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')


In [ ]:
# ============================================
# 4. FINAL TRAINING : OOF + Test Predictions
# ============================================
best_params = study.best_params
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'gpu_hist',
    'verbosity': 0,
})

N_REPEATS = 3  # 3 repeats × 5 folds = 15 models
oof_preds = np.zeros(len(X_train))
oof_counts = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
all_aucs = []

for repeat in range(N_REPEATS):
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=repeat)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(patients_unique, y_patient)):
        train_mask = np.isin(patients_train, patients_unique[train_idx])
        val_mask   = np.isin(patients_train, patients_unique[val_idx])
        
        X_tr, y_tr = X_train[train_mask], y_train[train_mask]
        X_val, y_val = X_train[val_mask], y_train[val_mask]
        
        model = xgb.XGBClassifier(**best_params, seed=42 + repeat * 100 + fold)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        val_preds = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, val_preds)
        all_aucs.append(auc)
        
        oof_preds[val_mask] += val_preds
        oof_counts[val_mask] += 1
        test_preds += model.predict_proba(X_test)[:, 1]
        
        print(f'  Repeat {repeat+1} Fold {fold+1}: AUC = {auc:.4f}')

# Average
oof_preds /= np.maximum(oof_counts, 1)
test_preds /= (N_REPEATS * N_FOLDS)

overall_oof_auc = roc_auc_score(y_train, oof_preds)
print(f'\n{"="*50}')
print(f'XGBoost Overall OOF AUC: {overall_oof_auc:.4f}')
print(f'Mean fold AUC: {np.mean(all_aucs):.4f} ± {np.std(all_aucs):.4f}')
print(f'Test pred mean: {test_preds.mean():.4f} | std: {test_preds.std():.4f}')
print(f'{"="*50}')


In [ ]:
# ============================================
# 5. SAVE TO DRIVE
# ============================================
# 5a. Individual submission
sub_xgb = pd.DataFrame({
    'Sample ID': df_meta_test['Sample ID'].values,
    'Target': test_preds
}).sort_values('Sample ID')

assert sub_xgb.shape == (149, 2)
assert all(sub_xgb['Target'].between(0, 1))

sub_xgb.to_csv(os.path.join(DRIVE_OUTPUT, 'submission_xgb.csv'), index=False)
print('✅ submission_xgb.csv saved')

# 5b. OOF predictions (for stacking)
np.save(os.path.join(DRIVE_OUTPUT, 'oof_xgb.npy'), oof_preds)
np.save(os.path.join(DRIVE_OUTPUT, 'test_xgb.npy'), test_preds)
print('✅ oof_xgb.npy + test_xgb.npy saved (for stacking)')

# 5c. Experiment params
experiment = {
    'model': 'xgboost',
    'best_params': best_params,
    'best_cv_auc': float(study.best_value),
    'overall_oof_auc': float(overall_oof_auc),
    'n_folds': N_FOLDS,
    'n_repeats': N_REPEATS,
    'n_optuna_trials': N_OPTUNA_TRIALS,
    'features': 'mean_pool_2048d',
}
with open(os.path.join(DRIVE_OUTPUT, 'params_xgb.json'), 'w') as f:
    json.dump(experiment, f, indent=4, default=str)
print('✅ params_xgb.json saved')

print(f'\n📁 All outputs in: {DRIVE_OUTPUT}')
print(sub_xgb.head())
